In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fe-tiep/__results__.html
/kaggle/input/fe-tiep/__notebook__.ipynb
/kaggle/input/fe-tiep/__output__.json
/kaggle/input/fe-tiep/custom.css
/kaggle/input/fe-tiep/feature_engineering/content_similarity.pkl
/kaggle/input/fe-tiep/feature_engineering/validation_data.parquet
/kaggle/input/fe-tiep/feature_engineering/cold_start_maps.pkl
/kaggle/input/fe-tiep/feature_engineering/customer_features.parquet
/kaggle/input/fe-tiep/feature_engineering/item_similarity_cosine.pkl
/kaggle/input/fe-tiep/feature_engineering/interaction_features.parquet
/kaggle/input/fe-tiep/feature_engineering/user_purchase_history.pkl
/kaggle/input/fe-tiep/feature_engineering/test_ground_truth.parquet
/kaggle/input/fe-tiep/feature_engineering/cooccurrence_topk.pkl
/kaggle/input/fe-tiep/feature_engineering/item_features.parquet


In [2]:
# ============================================================================
# NOTEBOOK 4: CANDIDATE GENERATION (STRICT SEPARATION STRATEGY)
# ============================================================================
# INPUTS:
# - user_purchase_history.pkl (Quyết định ai là Warm User)
# - cold_start_maps.pkl (Dùng để FILL cho Cold User)
# - customer_features.parquet (Lấy metadata Region/Gender)
#
# OUTPUTS:
# - user_candidates.pkl: Dictionary chứa items gợi ý cho tất cả user
# - user_segments.pkl: [QUAN TRỌNG] File chứa danh sách ID của 2 nhóm user
# ============================================================================

import polars as pl
import numpy as np
import pickle
import os
import gc
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("🚀 NOTEBOOK 4: CANDIDATE GENERATION")
print("   🎯 Strategy: SEPARATE WARM (Model) vs COLD (Heuristic)")
print("=" * 80)

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
BATCH_SIZE = 50_000      
MAX_CANDIDATES = 100   # Model sẽ rank 100 món này
TOP_K_CF = 20          
FORCE_TREND_COUNT = 20 

FEATURE_PATH = "/kaggle/input/fe-tiep/feature_engineering"
OUTPUT_PATH = "/kaggle/working/candidates"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ============================================================================
# 2. LOAD ARTIFACTS
# ============================================================================
print("\n[1/5] LOADING ARTIFACTS...")

def load_pickle(filename):
    path = f"{FEATURE_PATH}/{filename}"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"⚠ Warning: {filename} not found.")
    return {}

# Load dữ liệu từ NB3
user_history = load_pickle("user_purchase_history.pkl")
cold_start_maps = load_pickle("cold_start_maps.pkl")
cosine_cf = load_pickle("item_similarity_cosine.pkl")
implicit_cf = load_pickle("cooccurrence_topk.pkl")  

# Global Trends để backup
global_trends = cold_start_maps.get("global", [])[:100]

print(f"   ✓ History Users: {len(user_history):,}")
print(f"   ✓ Cold Start Maps: {list(cold_start_maps.keys())}")

# ============================================================================
# 3. SEGMENT USERS (WARM vs COLD)
# ============================================================================
print("\n[2/5] CLASSIFYING USERS (WARM vs COLD)...")

# Load tất cả user cần dự đoán (bao gồm cả user cũ và user mới trong tập test)
# Ở đây ta lấy từ customer_features (được build từ users table gốc)
df_users = pl.read_parquet(f"{FEATURE_PATH}/customer_features.parquet", columns=["customer_id", "region", "gender"])
all_users_info = df_users.to_dicts() 

# 1. Xác định Warm Users (Có lịch sử mua hàng)
warm_users_set = set(user_history.keys())

# 2. Tách danh sách
target_users_warm = []
target_users_cold = []
user_meta_map = {} # Map chỉ dành cho Cold user để tiết kiệm mem

for row in all_users_info:
    uid = row['customer_id']
    if uid in warm_users_set:
        target_users_warm.append(uid)
    else:
        target_users_cold.append(uid)
        # Chỉ lưu metadata cho cold user để dùng map
        user_meta_map[uid] = {
            'region': row.get('region', 'Unknown'),
            'gender': row.get('gender', 'Unknown')
        }

print(f"   📊 SEGMENTATION RESULT:")
print(f"   - Warm Users (No Cold Start): {len(target_users_warm):,} -> Will use History + CF + Model")
print(f"   - Cold Users (New/Feb Users): {len(target_users_cold):,} -> Will FILL with Popularity Rules")
print(f"   - Total Users:                {len(all_users_info):,}")

# ============================================================================
# 4. GENERATION LOGIC
# ============================================================================
print("\n[3/5] GENERATING CANDIDATES...")

all_candidates = {}

# --- A. XỬ LÝ WARM USERS (LOGIC PHỨC TẠP: REPURCHASE + CF) ---
print(f"   -> Processing Warm Users ({len(target_users_warm):,} users)...")

for i in range(0, len(target_users_warm), BATCH_SIZE):
    batch = target_users_warm[i : i + BATCH_SIZE]
    
    for user in batch:
        candidates = set()
        hist_items = user_history[user]
        
        # 1. REPURCHASE (Mấu chốt để tăng Precision)
        # Lấy 40 món mua gần nhất/nhiều nhất
        candidates.update(hist_items[:40])
        
        # 2. COLLABORATIVE FILTERING (Mở rộng để tăng Recall)
        # Lấy 3 món gần nhất làm 'seed'
        seeds = hist_items[:3]
        for seed in seeds:
            # Co-occurrence
            if seed in implicit_cf:
                for sim_item, _ in implicit_cf[seed][:TOP_K_CF]:
                    candidates.add(sim_item)
            # Cosine
            if seed in cosine_cf:
                for sim_item, _ in cosine_cf[seed][:10]:
                    candidates.add(sim_item)
                    
        # 3. GLOBAL FALLBACK (Nếu ít quá)
        if len(candidates) < MAX_CANDIDATES:
            candidates.update(global_trends[:FORCE_TREND_COUNT])
            
        all_candidates[user] = list(candidates)[:MAX_CANDIDATES]

# --- B. XỬ LÝ COLD USERS (LOGIC FILL: POPULARITY MAPS) ---
print(f"   -> Processing Cold Users ({len(target_users_cold):,} users)...")

# Pre-fetch maps để loop nhanh hơn
map_rg = cold_start_maps.get('region_gender', {})
map_r = cold_start_maps.get('region', {})
map_global = global_trends

for i in range(0, len(target_users_cold), BATCH_SIZE):
    batch = target_users_cold[i : i + BATCH_SIZE]
    
    for user in batch:
        # Cold users candidates là LIST (đã được sort theo độ phổ biến ở NB3)
        # Ta giữ nguyên thứ tự này để NB5 không cần rank lại cũng được
        candidates = [] 
        seen = set()
        
        meta = user_meta_map[user]
        r, g = meta['region'], meta['gender']
        
        # 1. Ưu tiên: Region + Gender
        key_rg = f"{r}_{g}"
        if key_rg in map_rg:
            for item in map_rg[key_rg]:
                if item not in seen:
                    candidates.append(item)
                    seen.add(item)
        
        # 2. Dự phòng: Region
        if len(candidates) < 50 and r in map_r:
            for item in map_r[r]:
                if item not in seen:
                    candidates.append(item)
                    seen.add(item)
                    
        # 3. Cuối cùng: Global
        for item in map_global:
            if len(candidates) >= MAX_CANDIDATES: break
            if item not in seen:
                candidates.append(item)
                seen.add(item)
                
        all_candidates[user] = candidates[:MAX_CANDIDATES]

print(f"   ✓ Generation Complete for {len(all_candidates):,} users.")

# ============================================================================
# 5. SAVE ARTIFACTS
# ============================================================================
print("\n[4/5] SAVING OUTPUTS...")

# 1. Save Candidates (Dictionary chung cho cả 2 nhóm)
with open(f"{OUTPUT_PATH}/user_candidates.pkl", "wb") as f:
    pickle.dump(all_candidates, f)

# 2. Save Segments (QUAN TRỌNG CHO NB6)
# NB6 sẽ load file này để biết tính mẫu số (denominator) cho đúng
segments = {
    "warm_users": target_users_warm, 
    "cold_users": target_users_cold,
    "all_users": target_users_warm + target_users_cold
}

with open(f"{OUTPUT_PATH}/user_segments.pkl", "wb") as f:
    pickle.dump(segments, f)

print(f"   ✓ Saved 'user_candidates.pkl' (Size: {len(all_candidates)})")
print(f"   ✓ Saved 'user_segments.pkl' (Keys: {list(segments.keys())})")

print("\n" + "="*80)
print("✅ NOTEBOOK 4 COMPLETED.")
print("👉 NEXT STEP (NB5):")
print("   - Warm Users: Chạy Feature Engineering + LightGBM Predict.")
print("   - Cold Users: Skip Model -> Dùng trực tiếp danh sách candidates (đã sort).")
print("="*80)

🚀 NOTEBOOK 4: CANDIDATE GENERATION
   🎯 Strategy: SEPARATE WARM (Model) vs COLD (Heuristic)

[1/5] LOADING ARTIFACTS...
   ✓ History Users: 1,153,598
   ✓ Cold Start Maps: ['global', 'region', 'region_gender', 'membership']

[2/5] CLASSIFYING USERS (WARM vs COLD)...
   📊 SEGMENTATION RESULT:
   - Warm Users (No Cold Start): 1,153,598 -> Will use History + CF + Model
   - Cold Users (New/Feb Users): 0 -> Will FILL with Popularity Rules
   - Total Users:                1,153,598

[3/5] GENERATING CANDIDATES...
   -> Processing Warm Users (1,153,598 users)...
   -> Processing Cold Users (0 users)...
   ✓ Generation Complete for 1,153,598 users.

[4/5] SAVING OUTPUTS...
   ✓ Saved 'user_candidates.pkl' (Size: 1153598)
   ✓ Saved 'user_segments.pkl' (Keys: ['warm_users', 'cold_users', 'all_users'])

✅ NOTEBOOK 4 COMPLETED.
👉 NEXT STEP (NB5):
   - Warm Users: Chạy Feature Engineering + LightGBM Predict.
   - Cold Users: Skip Model -> Dùng trực tiếp danh sách candidates (đã sort).
